In [1]:
import os
import json
import random

TRAIN_PATH = "project/data/train.jsonl"
TEST_PATH  = "project/data/test_30.jsonl"

In [2]:
print("Train exists:", os.path.exists(TRAIN_PATH))
print("Test exists :", os.path.exists(TEST_PATH))

train_lines = sum(1 for _ in open(TRAIN_PATH, "r", encoding="utf-8"))
test_lines  = sum(1 for _ in open(TEST_PATH, "r", encoding="utf-8"))

print("Train lines:", train_lines)
print("Test lines :", test_lines)


Train exists: True
Test exists : True
Train lines: 138
Test lines : 30


In [3]:
bad = 0

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        try:
            obj = json.loads(line)
            assert "instruction" in obj
            assert "response" in obj
            assert "short_answer" in obj["response"]
            assert "confidence_level" in obj["response"]
        except Exception as e:
            bad += 1
            print(f"Bad TRAIN line {i}: {e}")

print("Bad TRAIN lines:", bad)

bad = 0
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        try:
            obj = json.loads(line)
            assert "instruction" in obj
            assert len(obj.keys()) == 1  # só pergunta no test set
        except Exception as e:
            bad += 1
            print(f"Bad TEST line {i}: {e}")

print("Bad TEST lines:", bad)


Bad TRAIN lines: 0
Bad TEST lines: 0


In [4]:
train = [json.loads(l) for l in open(TRAIN_PATH, "r", encoding="utf-8").read().splitlines()]
test  = [json.loads(l) for l in open(TEST_PATH, "r", encoding="utf-8").read().splitlines()]

print("Random TRAIN samples:")
for ex in random.sample(train, 3):
    print("\nQ:", ex["instruction"])
    print("A:", ex["response"]["short_answer"])
    print("Conf:", ex["response"]["confidence_level"])
    print("Note:", ex["response"].get("clinical_notes",""))

print("\nRandom TEST samples:")
for ex in random.sample(test, 3):
    print("-", ex["instruction"])


Random TRAIN samples:

Q: What is mTOR and why is it discussed in longevity research?
A: mTOR is a nutrient-sensing signaling pathway involved in growth, metabolism, and protein synthesis; it is studied in longevity because altering mTOR activity affects aging-related processes in multiple model organisms.
Conf: high
Note: Translating mechanistic findings into safe human longevity interventions remains uncertain.

Q: What is epigenetic aging and what do 'epigenetic clocks' measure?
A: Epigenetic clocks estimate biological age using DNA methylation patterns that correlate with chronological age and some health outcomes.
Conf: high
Note: They are useful in research but are not universally validated for individual clinical decision-making.

Q: Do cardiovascular benefits of GLP-1 receptor agonists necessarily require weight loss?
A: Some cardiovascular benefits appear partly independent of weight loss, but mechanisms and the extent of independence vary by drug and population.
Conf: medium


In [5]:
from collections import Counter

conf = Counter(ex["response"]["confidence_level"] for ex in train)
print("Confidence distribution:", conf)


Confidence distribution: Counter({'high': 103, 'medium': 35})


In [6]:
summary = {
    "train_lines": train_lines,
    "test_lines": test_lines,
    "train_confidence_distribution": dict(conf),
    "bad_train_lines": 0,
    "bad_test_lines": 0
}

os.makedirs("project/report", exist_ok=True)
with open("project/report/inspection_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Wrote project/report/inspection_summary.json")


Wrote project/report/inspection_summary.json
